## Error de reconstrucción por clase

Se usaron 1000 imagenes del dataset MNIST para evaluar el error de reconstrucción por dígito de cada modelo.
Se usan las imagenes originales y la reconstrución para calcular el error cuadratico medio entre ambas figuras.

In [1]:
import numpy as np
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from collections import defaultdict


def evaluar_reconstruccion_por_clase(cvae, x_test, y_test, metric="mse"):
    """
    Evalúa la calidad de reconstrucción para cada clase.

    Args:
        cvae: modelo CVAE entrenado
        x_test: imágenes de prueba (shape: [N, 28, 28])
        y_test: etiquetas one-hot (shape: [N, 10])
        metric: 'mse' o 'bce'

    Returns:
        dict: promedio del error por clase
    """
    errores = defaultdict(list)

    for i in tqdm(range(len(x_test))):
        x = x_test[i : i + 1]  # (1, 28, 28)
        y = y_test[i : i + 1]  # (1, 10)

        # Codificar
        z_mean, _, z = cvae.encoder.predict([x, y], verbose=0)
        # Decodificar
        x_recon = cvae.decoder.predict([z, y], verbose=0)

        # Flatten para comparar
        x_flat = x.flatten()
        x_recon_flat = x_recon.flatten()

        if metric == "mse":
            error = mean_squared_error(x_flat, x_recon_flat)
        elif metric == "bce":
            import tensorflow.keras.losses as losses

            error = losses.binary_crossentropy(x_flat, x_recon_flat).numpy().mean()

        clase = np.argmax(y)
        errores[clase].append(error)

    # Promedio por clase
    resultados = {clase: np.mean(errores[clase]) for clase in errores}
    return resultados

In [2]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

from project.trained_models import load

dataset = "mnist" # cambiar aca

data = load.data(dataset=dataset)
models = load.all_models(dataset=dataset)

x_test = data["x_test"]
y_test = data["y_test"]

resultados = []


for m in models:
    resultados.append(evaluar_reconstruccion_por_clase(m, x_test[0:1000], y_test[0:1000], metric="mse"))

for i in range(len(models)):
    print(f"{models[i].name}")
    for clase, error in sorted(resultados[i].items()):
        print(f" Dígito {clase}: error promedio = {error:.3f}")


2025-11-26 11:09:20.240001: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-26 11:09:20.243356: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-26 11:09:20.253346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764166160.270138 2040606 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764166160.275025 2040606 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764166160.288767 2040606 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

Usando mnist como dataset
Encontrados 8 pares de modelos.


2025-11-26 11:09:24.001353: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_UNKNOWN: unknown error
2025-11-26 11:09:24.001373: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-11-26 11:09:24.001378: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: pc-santi
2025-11-26 11:09:24.001382: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:190] hostname: pc-santi
2025-11-26 11:09:24.001492: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:197] libcuda reported version is: 575.64.3
2025-11-26 11:09:24.001510: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:201] kernel reported version is: 575.64.3
2025-

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt


def plot_error(resultados, ruta=None,title=""):
    valores = list(resultados.values())
    claves = list(resultados.keys())

    # Graficar barras
    bars = plt.bar(claves, valores)

    # Etiquetas encima de cada barra
    for i, bar in enumerate(bars):
        altura = bar.get_height()
        color_fondo = bar.get_facecolor()

        # Elegir blanco o negro según luminosidad del color de la barra
        r, g, b, _ = color_fondo
        color_texto = "black"

        plt.text(
            bar.get_x() + bar.get_width() / 2,  # centrado horizontal
            altura + 0.0015,  # un poco abajo del tope de la barra
            f"{valores[i]:.3f}",
            ha="center",
            va="top",
            color=color_texto,
            fontsize=10,
        )

    plt.xlabel("Dígito")
    plt.ylabel("Error de reconstrucción [MSE]")
    plt.title(f"Reconstrucción promedio por clase modelo: {title}")
    plt.tight_layout()
    if ruta:
        plt.savefig(f"figs/{ruta}.png")
    plt.show()


"""

"""

In [ ]:
for m,r in zip(models,resultados):
    plot_error(resultados=r, ruta=m.name,title=m.name)




quizas el error tiene que ver con la variabilidad de cada dígito ("mas formas de dibujar un 8 que mas formas de dibjuar un 1")
¿Como aprovehcar esto para mejorar el aprendizaje?
tal vez se pueda ponderar la loss de reconstruccion en funcion del dígito 
coeficinetes (al principio en 1)->una epoca-> validacion comparando contra estos valores-> ajuste-> epocas---> se "aplana"  este grafico 
el error.

Ambos modelos se comportan de maneras muy similiar, practicamente identicos. --> ¿Es necesario tener modelos tan grandes para esta tarea?

- El bajo error en la reconstrucción del dígito 1 ¿Tendra que ver con la poca "dispersión o variabilidad" para dibujar el número? --> El numero 2 y 8 son los que más error tienen, ¿Más formas de dibujar estos números?


Con el dataset de fashion-mnist. el error es mas parejo. La clase con menor error la clase de patalones, mientras que la que tiene más error de reconstrucción es la calse de bolsos. Todos los modelos tiene una distribución similar en cuanto en cuanto al error (En ambos datasets).  
No parece haber una mejora de la reconstrucción con más parametros.

## Prueba con más capas

ponemos 2 capas de 64 en lugar de una de 128 para evaluar el error de reconstrucción

In [ ]:
from keras.layers import Input, Dense, Concatenate, Reshape
from keras.models import Model
from custom_layers.Sampling import Sampling
from models_definitions.cvae import CVAE 
import tensorflow as tf
from keras.callbacks import EarlyStopping
from experiments import load


condition_dim=(10,)
intermediate_dim=128
latent_dim=2
img_dim=(28,28)
flat_dim = img_dim[0]*img_dim[1]
img_input = Input(shape=(flat_dim,), name="img_input_encoder")
cond_encoder = Input(shape=(condition_dim), name="encoder_condition")
imputs_cocanteados = Concatenate()([img_input, cond_encoder])


x = Dense(96, activation="relu")(imputs_cocanteados) #  96+32=128 un paso intermedio, un escalon más
x = Dense(32, activation="relu")(x) #

z_mean = Dense(latent_dim, name="z_mean")(x)
z_log_var = Dense(latent_dim, name="z_log_var")(x)
z = Sampling()((z_mean, z_log_var))
encoder = Model(inputs=[img_input, cond_encoder], outputs=[z_mean, z_log_var, z], name="encoder")



latent_dim = 2
cond_dim=(10,)
intermediate_dim=128
original_shape=(28, 28)
original_dim = original_shape[0] * original_shape[1]

z_inputs = Input(shape=(latent_dim,), name="z_sampling")
cond_decoder = Input(shape=cond_dim, name="decoder_condition")

latent_inputs = Concatenate()([z_inputs, cond_decoder])

x = Dense(32, activation="relu")(latent_inputs)
x = Dense(96, activation="relu")(x)

decoder_outputs = Dense(original_dim, activation="sigmoid")(x)

decoder = Model(inputs=[z_inputs, cond_decoder], outputs=decoder_outputs, name="decoder")


original_dim = 28 * 28
beta = 1.0

data = load.data("fashion")
x_train =  data["x_train"]
y_train = data["y_train"]
x_test =  data["x_test"]
y_test = data["y_test"]
x_val = data["x_val"]
y_val = data["y_val"]



train_dataset = tf.data.Dataset.from_tensor_slices(((x_train, y_train), x_train))
# train_dataset = train_dataset.shuffle(buffer_size=1024).batch(128)
train_dataset = train_dataset.batch(128)

early_stopping = EarlyStopping(
    monitor="val_loss",  # Monitor validation loss
    patience=3,  # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True,  # Restore model weights from the epoch with the best value of the monitored quantity
)

val_dataset = tf.data.Dataset.from_tensor_slices(((x_val, y_val), x_val))
val_dataset = val_dataset.batch(128)


cvae = CVAE(encoder=encoder, decoder=decoder, original_dim=original_dim, beta=1)
cvae.compile(optimizer=tf.keras.optimizers.Adam())

cvae.fit(
    train_dataset,
    epochs=1000,
    batch_size=128,
    validation_data=val_dataset,
    callbacks=[early_stopping],
)


resultados_mas_pasos = evaluar_reconstruccion_por_clase(cvae, x_test[0:1000], y_test[0:1000], metric="mse")


print(f"{cvae.name}")
for clase, error in sorted(resultados_mas_pasos.items()):
    print(f" Dígito {clase}: error promedio = {error:.3f}")


In [ ]:
plot_error(resultados_mas_pasos, ruta="cvae_96_32_2_fashion",title=cvae.name)